# RQ3: Decomposition Branching Evaluation

This notebook compares two branching approaches for the Branch and Bound solver:

- **Indicator Modeling**: Uses binary indicator variables in the SOCP formulation to model which polygon boundary each visit point lies on
- **Decomposition Branching**: Explicitly branches on polygon decompositions, creating child nodes for each possible boundary segment

We evaluate performance across three instance types:
- **OSM**: Real-world instances derived from OpenStreetMap data
- **Tessellation**: Synthetic instances from Voronoi tessellations
- **Random**: Randomly generated polygon instances

## Setup and Data Loading

Load benchmark results comparing decomposition branching (True) vs indicator modeling (False).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from algbench import read_as_pandas
from matplotlib.ticker import PercentFormatter
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, init_params
from tspn_bnb2.misc.plotting import cactus_plot
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    if row["result"]["solution"] is None:
        print("Not found", row["parameters"]["args"]["kwargs"]["instance_name"])
        return None
    solution = AnnotatedSolution.model_validate_json(row["result"]["solution"])

    skip_convex_hull = row["parameters"]["args"]["alg_params"]["skip_convex_hull"]
    decomposition = row["parameters"]["args"]["alg_params"]["decomposition"]

    if skip_convex_hull and not decomposition:
        strategy = "eager indicator\nmodeling"
    elif decomposition:
        strategy = "decomposition\nbranching"
    else:
        assert not decomposition and not skip_convex_hull
        strategy = "indicator\nmodeling"

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "gap": (solution.upper_bound / solution.lower_bound) - 1,
        "relative_gap": solution.relative_gap,
        "is_optimal": solution.is_optimal,
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": instance,
        "solve_time": row["result"]["solve_time"],
        "n": instance.num_polygons(),
        "decomposition": decomposition,
        "skip_convex_hull": skip_convex_hull,
        "decomposition_strategy": strategy,
        "instance_type": instance.derive_instance_type(),
    }


benchmark = read_as_pandas("results_decomposition_strategy", read_row)
print("Loaded", len(benchmark), "runs")
benchmark = benchmark.sort_values(by=["decomposition_strategy", "instance_type"])

In [ ]:
benchmark[benchmark["upper_bound"] == float("inf")]

## Solution Validation

Cross-check that solutions from both approaches are consistent. For each instance solved by both methods, verify that the bounds agree within the optimality tolerance.

In [ ]:
# validate that solutions are correct.
n_checks = 0
for _, row in benchmark.iterrows():
    solution = row["solution"]
    if solution is None:
        continue
    same_instance = benchmark[benchmark["instance_name"] == row["instance_name"]]

    for _, other in same_instance.iterrows():
        if other["solution"] is None:
            continue
        check = solution.plausibility_check(
            other["solution"],
            eps=0.01,
        )
        if not check["valid"]:
            print(
                f"Check failed for {row['instance_name']}"
                f" {check} {row['solution']}\n"
                f"{row['decomposition']}\n"
                f"{other['solution']}\n"
                f"{other['decomposition']}"
            )
        else:
            n_checks += 1

print(n_checks, "checks succeeded")

## Optimality by Strategy and Instance Type

Count how many instances each approach solved to optimality, broken down by instance type.

In [ ]:
benchmark.groupby(["decomposition_strategy", "instance_type"])["is_optimal"].value_counts()

## Runtime Comparison

**Left plot**: Absolute runtime comparison between decomposition branching and indicator modeling, grouped by instance type. Percentages show the fraction of instances solved to optimality.

**Right plot**: Relative runtime (decomposition branching / indicator modeling) by instance type. Values below 1.0 indicate decomposition branching is faster.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

ax = axs[0]
sns.boxplot(
    benchmark,
    x="decomposition_strategy",
    y="solve_time",
    hue="instance_type",
    palette=sns.color_palette(),
    ax=ax,
    hue_order=["OSM", "tessellation", "random"],
)

xticks = list(ax.get_xticks())
xticklabels = []

for label in ax.get_xticklabels():
    decomp = label.get_text()

    solutions_for_label = benchmark[(benchmark["decomposition_strategy"] == decomp)]
    optimal_percentage = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = decomp + f"\n[{round(optimal_percentage * 100, 2)}\\%]"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)

ax.set_title("lower is better")
ax.set_xlabel("decomposition")
ax.set_ylabel("runtime [s]")

instances_with_all_opt = [
    inst
    for inst in benchmark["instance_name"].unique()
    if len(benchmark[(benchmark["instance_name"] == inst)]) == 3
]
bench = benchmark[benchmark["instance_name"].isin(instances_with_all_opt)]

df = {"instance": [], "instance_type": [], "relative_runtime": []}

for _, row in bench[bench["decomposition"]].iterrows():
    other = bench[
        (bench["decomposition_strategy"] != row["decomposition_strategy"])
        & (bench["instance_name"] == row["instance_name"])
    ]
    assert len(other) == 2, len(other)

    for _, other_row in other.iterrows():
        df["instance"].append(row["instance_name"])
        df["instance_type"].append(row["instance_type"])
        df["relative_runtime"].append(row["solve_time"] / other_row["solve_time"])

df = pd.DataFrame(df)
df = df.sort_values(
    "instance_type",
    key=lambda x: pd.Categorical(x, categories=["OSM", "tessellation", "random"], ordered=True),
)

print(len(df))
sns.boxplot(
    df,
    x="instance_type",
    y="relative_runtime",
    hue="instance_type",
    ax=axs[1],
    hue_order=["OSM", "tessellation", "random"],
)

axs[1].set_ylabel("relative runtime [s]")
axs[1].set_title("lower is better for DB")
axs[1].set_xlabel("")
axs[0].set_xlabel("")

fig.legend(loc="outside lower center", ncol=3, bbox_to_anchor=(0.5, -0.32))
axs[0].legend().remove()
fig.subplots_adjust(wspace=0.3)
fig.savefig("../plots/rq3_decomposition_branching/runtimes_all.pdf", bbox_inches="tight")

## Detailed Performance Analysis

**Left (Cactus plot)**: For instances where exploration counts differ, shows how many instances are solved over time. Higher curves indicate better performance.

**Right (Gap distribution)**: For instances not solved to optimality, shows the remaining optimality gap for each approach.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

# instance_count = len(set(benchmark["instance_name"].unique()))

bm = benchmark[(benchmark["is_optimal"])]

df = cactus_plot(
    bm,
    ax=axs[0],
    instance_column="instance_name",
    strategy_column="decomposition_strategy",
    runtime_column="solve_time",
    # flat_line_to=benchmark["solve_time"].max(),
)
axs[0].legend()
# axs[0].set_xlim(0, axs[0].get_xlim()[1])


other_bm = benchmark[(~benchmark["is_optimal"])]

sns.boxplot(other_bm, x="decomposition_strategy", y="gap", ax=axs[1], hue="decomposition_strategy")

axs[1].yaxis.set_major_formatter(PercentFormatter(1.0))

axs[0].set_title(f"{len(bm['instance_name'].unique())} instances solved to optimality")
axs[1].set_title(f"{len(other_bm['instance_name'].unique())} instances with gap")

# fig.subplots_adjust(right=0.74)
fig.savefig("../plots/rq3_decomposition_branching/cactus.pdf", bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=FULLWIDEFIGURE)

colors = {v: f"C{i}" for i, v in enumerate(benchmark["decomposition_strategy"].unique())}

benchmark_solved = benchmark[(benchmark["upper_bound"] != float("inf"))]
instance_types = [["random", "tessellation"], "OSM"]

for i, (ax, instance_type) in enumerate(zip(axs, instance_types)):
    if isinstance(instance_type, list):
        solved = benchmark_solved[benchmark_solved["instance_type"].isin(instance_type)]
    else:
        solved = benchmark_solved[benchmark_solved["instance_type"] == instance_type]

    max_solve_time_for_type = solved["solve_time"].max()

    print(max_solve_time_for_type)

    cactus_plot(
        solved[solved["is_optimal"]],
        ax=ax,
        instance_column="instance_name",
        strategy_column="decomposition_strategy",
        runtime_column="solve_time",
        flat_line_to=max_solve_time_for_type,
        colors=colors,
        linestyle="-",
    )
    ymax = ax.get_ylim()[1]
    ax.set_ylim(ymax * 0.5, ymax)
    ax.set_xlim(0, max_solve_time_for_type)

    if i != 0:
        ax.legend().remove()
        ax.set_ylabel("")

    ax.set_title(", ".join(instance_type) if isinstance(instance_type, list) else instance_type)

solved = benchmark_solved[benchmark_solved["instance_type"] == "OSM"]
max_gap_for_type = np.nanmax(solved["gap"])
print(solved["lower_bound"].min())
print(solved["upper_bound"].max())
cactus_plot(
    solved,
    ax=axs[2],
    instance_column="instance_name",
    strategy_column="decomposition_strategy",
    runtime_column="gap",
    flat_line_to=max_gap_for_type,
    colors=colors,
    linestyle="-",
)

axs[2].set_xlim(0.01, max_gap_for_type)
axs[2].set_xlabel(r"gap [\%]")
axs[2].xaxis.set_major_formatter(PercentFormatter(1.0))
axs[2].set_ylim(axs[1].get_ylim())
axs[2].set_title("OSM")
axs[2].set_ylabel("")

fig.suptitle("higher is better")
fig.subplots_adjust(top=0.8, wspace=0.3)
fig.savefig("../plots/rq3_decomposition_branching/cactus_by_instance_type.pdf", facecolor="white")

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

colors = {v: f"C{i}" for i, v in enumerate(benchmark["decomposition_strategy"].unique())}

instance_types = ["random", "OSM"]
benchmark_solved = benchmark[
    (benchmark["instance_type"].isin(instance_types)) & (benchmark["upper_bound"] != float("inf"))
]


cactus_plot(
    benchmark_solved[benchmark_solved["is_optimal"]],
    ax=axs[0],
    instance_column="instance_name",
    strategy_column="decomposition_strategy",
    runtime_column="solve_time",
    flat_line_to=60,
    colors=colors,
    linestyle="-",
)

cactus_plot(
    benchmark_solved,
    ax=axs[1],
    instance_column="instance_name",
    strategy_column="decomposition_strategy",
    runtime_column="gap",
    flat_line_to=60,
    colors=colors,
    linestyle="-",
)


axs[0].set_xlim(0, 60)
axs[1].set_xlim(0.01, benchmark_solved["gap"].max())
axs[1].xaxis.set_major_formatter(PercentFormatter(1.0, symbol="", decimals=0))
axs[1].set_xticks([0.01, 0.1, 0.2, 0.3, 0.4])
axs[1].set_xlabel(r"gap [\%]")

axs[0].legend(loc="lower right")
axs[0].set_title("higher is better")
axs[1].set_title("higher is better")


n_instances = benchmark_solved["instance_name"].nunique()
print(n_instances, "instances")

yticks = np.linspace(0, n_instances, 11)  # 0%, 10%, ..., 100%

axs[0].set_ylabel(r"solved instances [\%]")
axs[1].set_ylabel("")

for ax in axs:
    ax.yaxis.set_major_formatter(PercentFormatter(n_instances, symbol=""))
    ax.set_yticks(yticks)
    ax.set_ylim(n_instances * 0.4, n_instances * 1.05)

fig.subplots_adjust(top=0.8, wspace=0.3)
fig.savefig("../plots/rq3_decomposition_branching/cactus_decomp_im_eim.pdf", facecolor="white")

In [ ]:
benchmark[benchmark["instance_type"].isin(["OSM", "random"])].groupby(["decomposition_strategy"])[
    "is_optimal"
].mean()